# Project 8: Long-Term Memory Agent

Compare recent-window, episodic, and hybrid semantic/temporal memory. Test
corrections, conflicts, evidence, consolidation, and deletion across all stores.

In [1]:
from pathlib import Path
import os,subprocess,sys
candidates=[Path.cwd(),Path.cwd()/"project8",Path("/content/ai_agentic_attemptings/project8")]
PROJECT_ROOT=next((p.resolve() for p in candidates if (p/"config/default.json").exists()),None)
if PROJECT_ROOT is None:
    repo=Path("/content/ai_agentic_attemptings")
    if not repo.exists(): subprocess.run(["git","clone","https://github.com/soraber/ai_agentic_attemptings.git",str(repo)],check=True)
    else: subprocess.run(["git","-C",str(repo),"pull","--ff-only"],check=True)
    PROJECT_ROOT=repo/"project8"
os.chdir(PROJECT_ROOT)
if not os.getenv("AI_PROJECT_SKIP_INSTALL"):
    subprocess.run([sys.executable,"-m","pip","install","--upgrade-strategy","only-if-needed","-r","requirements-colab.txt"],check=True)
    subprocess.run([sys.executable,"-m","pip","install","-e",".","--no-deps"],check=True)
source_root=PROJECT_ROOT/"src"
if str(source_root) not in sys.path: sys.path.insert(0,str(source_root))
if not os.getenv("AI_PROJECT_SKIP_INSTALL"):
    check=subprocess.run([sys.executable,"-m","pip","check"],text=True,capture_output=True)
    if check.returncode: print(check.stdout or check.stderr)
from project8_agent.memory import MemoryStore
try: import torch
except ImportError: torch=None
print("Project 8 imports passed")
print({"cuda":bool(torch and torch.cuda.is_available()),"gpu":torch.cuda.get_device_name(0) if torch and torch.cuda.is_available() else None})

ipython 7.34.0 requires jedi, which is not installed.
ibis-framework 9.5.0 has requirement sqlglot<25.21,>=23.4, but you have sqlglot 29.0.1.



Project 8 imports passed
{'cuda': True, 'gpu': 'NVIDIA A100-SXM4-40GB'}


In [2]:
import getpass,os,sys
from project8_agent.config import load_config
EVAL_BACKEND="local_gpu"  # deterministic | openai | local_gpu
RUN_FULL_EVAL=True; RUN_API_EVAL=EVAL_BACKEND=="openai"; RUN_LOCAL_GPU_EVAL=EVAL_BACKEND=="local_gpu"; config=load_config(PROJECT_ROOT/"config/default.json")
if RUN_API_EVAL and not os.getenv("OPENAI_API_KEY"):
    if "google.colab" in sys.modules:
        from google.colab import userdata
        key=userdata.get("OPENAI_API_KEY")
    else: key=getpass.getpass("OPENAI_API_KEY (hidden): ")
    if not key: raise RuntimeError("OPENAI_API_KEY required for API mode")
    os.environ["OPENAI_API_KEY"]=key
if RUN_LOCAL_GPU_EVAL:
    import gc,torch
    if not torch.cuda.is_available(): raise RuntimeError("Select a Colab GPU runtime for local_gpu mode")
    stale_names=("source_planner","schema_retriever","local_backend","planners","planner","embedding_retriever","local_answerer")
    released=[name for name in stale_names if globals().pop(name,None) is not None]
    gc.collect(); torch.cuda.empty_cache(); free_bytes,total_bytes=torch.cuda.mem_get_info()
    print({"released_gpu_objects":released,"free_gpu_gib":round(free_bytes/2**30,2),"total_gpu_gib":round(total_bytes/2**30,2)})
print(config.model_dump())

{'released_gpu_objects': [], 'free_gpu_gib': 39.08, 'total_gpu_gib': 39.49}
{'project_id': 'project8', 'seed': 20260802, 'model': 'gpt-5.6-luna', 'reasoning_effort': 'low', 'working_window_size': 6, 'episodic_top_k': 5, 'qa_per_conversation': 40, 'selected_conversations': 2, 'max_model_calls': 300, 'max_output_tokens': 500, 'max_retries': 2, 'max_estimated_cost_usd': 8.0, 'input_price_per_million_usd': 1.0, 'output_price_per_million_usd': 6.0, 'local_model': 'Qwen/Qwen2.5-7B-Instruct', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'local_device': 'cuda', 'local_max_new_tokens': 300}


In [3]:
import subprocess,sys
subprocess.run([sys.executable,"tools/fetch_locomo.py"],check=True)
subset_path=PROJECT_ROOT/"data/cache/locomo_subset.json"
print("LoCoMo subset:",subset_path.relative_to(PROJECT_ROOT).as_posix())

LoCoMo subset: data/cache/locomo_subset.json


In [4]:
import json
subset=json.loads(subset_path.read_text()); lifecycle=json.loads((PROJECT_ROOT/"data/lifecycle_cases.json").read_text())
assert len(subset)==2 and sum(len(item["qa"]) for item in subset)==80
print({"conversations":2,"qa":80,"lifecycle_events":len(lifecycle["events"])})

{'conversations': 2, 'qa': 80, 'lifecycle_events': 7}


In [5]:
from project8_agent.memory import MemoryStore
from project8_agent.schemas import MemoryEvent,MemoryQuery
runtime=PROJECT_ROOT/"output/runtime"; runtime.mkdir(parents=True,exist_ok=True); store=MemoryStore(runtime/"memory.sqlite"); store.reset()
for item in lifecycle["events"]: store.ingest(MemoryEvent.model_validate(item))
query=MemoryQuery.model_validate(lifecycle["queries"][0]); print(store.answer(query,"window").model_dump()); print(store.answer(query,"episodic").model_dump())
embedding_retriever=None
if RUN_LOCAL_GPU_EVAL:
    from project8_agent.local_models import EmbeddingEventRetriever
    from project8_agent.locomo import conversation_events
    embedding_retriever=EmbeddingEventRetriever(config.embedding_model,config.local_device)
    sample_events=conversation_events(subset[0]); print({"dense_retrieval":[item["event_id"] for item in embedding_retriever.retrieve(sample_events,subset[0]["qa"][0]["question"],"hybrid",config.working_window_size,config.episodic_top_k)]})

{'query_id': 'Q01', 'system': 'window', 'answer': None, 'evidence_ids': [], 'abstained': True, 'conflict': False, 'context_tokens': 12, 'latency_ms': 0.3891180003847694}
{'query_id': 'Q01', 'system': 'episodic', 'answer': 'Boston', 'evidence_ids': ['E01'], 'abstained': False, 'conflict': False, 'context_tokens': 2, 'latency_ms': 0.6362590011121938}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'dense_retrieval': ['D9:11', 'D14:34', 'D1:3', 'D5:2', 'D8:22']}


In [6]:
for event_id in lifecycle["delete_event_ids"]: store.delete_event(event_id)
correction=MemoryQuery.model_validate(lifecycle["queries"][1]); conflict=MemoryQuery.model_validate(lifecycle["queries"][2])
print(store.answer(correction,"hybrid").model_dump()); print(store.answer(conflict,"hybrid").model_dump()); assert all(store.deletion_verified(e) for e in lifecycle["delete_event_ids"])

{'query_id': 'Q02', 'system': 'hybrid', 'answer': 'green', 'evidence_ids': ['E04'], 'abstained': False, 'conflict': False, 'context_tokens': 4, 'latency_ms': 0.7655110002815491}
{'query_id': 'Q03', 'system': 'hybrid', 'answer': None, 'evidence_ids': [], 'abstained': True, 'conflict': True, 'context_tokens': 4, 'latency_ms': 0.6937289999768836}


In [7]:
import os,subprocess,sys
test_env=os.environ.copy(); test_env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"]="1"
try:
    result=subprocess.run([sys.executable,"-m","pytest","-q","tests"],text=True,capture_output=True,env=test_env,timeout=120)
except subprocess.TimeoutExpired as exc:
    raise RuntimeError("Project 8 tests exceeded the 120-second Colab limit") from exc
print(result.stdout)
if result.returncode: print(result.stderr); raise RuntimeError("Project 8 tests failed")

............                                                             [100%]
12 passed in 0.53s



In [8]:
from project8_agent.evaluation import evaluate_lifecycle
from project8_agent.locomo import CachedLocalAnswerer, evaluate_locomo
if not RUN_FULL_EVAL: print("Set RUN_FULL_EVAL=True after P08-C07 passes.")
else:
    lifecycle_summary=evaluate_lifecycle(PROJECT_ROOT/"data/lifecycle_cases.json",runtime/"evaluation.sqlite",runtime/"lifecycle_output")
    if RUN_LOCAL_GPU_EVAL:
        result_dir=PROJECT_ROOT/"output/gpu"; local_answerer=CachedLocalAnswerer(config,runtime/"locomo_local_answer_cache.json")
        summary=evaluate_locomo(subset_path,config,runtime/"unused.json",result_dir,lifecycle_summary,answerer=local_answerer,retriever=embedding_retriever,evaluation_mode="locomo_local_gpu")
    elif RUN_API_EVAL:
        summary=evaluate_locomo(subset_path,config,runtime/"locomo_answer_cache.json",PROJECT_ROOT/"output",lifecycle_summary)
    else:
        summary=evaluate_lifecycle(PROJECT_ROOT/"data/lifecycle_cases.json",runtime/"evaluation.sqlite",PROJECT_ROOT/"output")
    print(summary)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

{'project': 'Long-Term Memory Agent', 'result_status': 'measured', 'evaluation_mode': 'locomo_local_gpu', 'qa_items': 80, 'window': {'exact_match_pct': 0.0, 'mean_token_f1': 0.006377857035751773, 'mean_evidence_recall': 0.0125, 'mean_context_tokens': 90.5}, 'episodic': {'exact_match_pct': 0.0, 'mean_token_f1': 0.1000041206882756, 'mean_evidence_recall': 0.15, 'mean_context_tokens': 91.1}, 'hybrid': {'exact_match_pct': 0.0, 'mean_token_f1': 0.06631429357699108, 'mean_evidence_recall': 0.13125, 'mean_context_tokens': 87.25}, 'model': 'Qwen/Qwen2.5-7B-Instruct', 'retrieval_model': 'sentence-transformers/all-MiniLM-L6-v2', 'model_calls': 150, 'input_tokens': 69462, 'output_tokens': 6426, 'estimated_cost_usd': 0.0, 'local_model': 'Qwen/Qwen2.5-7B-Instruct', 'device': 'cuda', 'cache_hits': 90, 'deletion_compliance_pct': 100.0, 'lifecycle_validation': {'project': 'Long-Term Memory Agent', 'result_status': 'measured', 'window': {'exact_match_pct': 75.0, 'mean_token_f1': 0.75, 'mean_evidence_re

In [9]:
import json
result_dir=PROJECT_ROOT/("output/gpu" if RUN_LOCAL_GPU_EVAL else "output")
path=result_dir/"project8_representative_samples.json"; print(json.loads(path.read_text()) if path.exists() else "Run P08-C08 first.")

{'best_hybrid': [], 'hybrid_failures': [{'answer': 'Caroline went to the LGBTQ support group on 8 May, 2023.', 'context_tokens': 108, 'evidence_ids': ['D1:3'], 'evidence_recall': 1.0, 'exact_match': False, 'gold': '7 May 2023', 'qa_index': 0, 'question': 'When did Caroline go to the LGBTQ support group?', 'sample_id': 'conv-26', 'system': 'hybrid', 'token_f1': 0.28571428571428575}, {'answer': 'Last year', 'context_tokens': 79, 'evidence_ids': ['D1:14'], 'evidence_recall': 0.0, 'exact_match': False, 'gold': '2022', 'qa_index': 1, 'question': 'When did Melanie paint a sunrise?', 'sample_id': 'conv-26', 'system': 'hybrid', 'token_f1': 0.0}, {'answer': 'The provided context does not give specific information about the fields Caroline would likely pursue in her education. The conversation focuses more on general topics such as jobs, life experiences, art, and community service.', 'context_tokens': 77, 'evidence_ids': [], 'evidence_recall': 0.0, 'exact_match': False, 'gold': 'Psychology, cou

In [10]:
import subprocess,sys
result_dir=PROJECT_ROOT/("output/gpu" if RUN_LOCAL_GPU_EVAL else "output"); summary_path=result_dir/"project8_final_summary.json"
if summary_path.exists():
    subprocess.run([sys.executable,"tools/generate_report.py","--summary",str(summary_path),"--output",str(result_dir/"project8_report.pdf")],check=True); subprocess.run([sys.executable,"tools/validate_project.py"]+([] if RUN_LOCAL_GPU_EVAL else ["--require-results"]),check=True)
else: print("Measured summary absent; report generation skipped.")